In [1]:
import os

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

In [2]:
import pandas as pd

ft_model_name = 'Qwen3-14B'

data_root = '/tmp/code/transformers_study/whtcc_01_Qwen3_SFT_Study/datasets/huanhuan_data/huanhuan.json'

df = pd.read_json(data_root)

In [3]:
from datasets import Dataset

datas = Dataset.from_pandas(df)
datas

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 3729
})

In [4]:
datas = datas.train_test_split(test_size=0.2)
datas

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 2983
    })
    test: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 746
    })
})

In [6]:
import torch
from transformers import Trainer, AutoModelForCausalLM, AutoTokenizer, DataCollatorForSeq2Seq, TrainingArguments

qwen3_model_name = f'/tmp/pretrainmodel/{ft_model_name}'

tokenizer = AutoTokenizer.from_pretrained(qwen3_model_name, use_fast=False, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(qwen3_model_name, device_map='auto', torch_dtype=torch.bfloat16)

Loading checkpoint shards: 100%|██████████| 8/8 [01:24<00:00, 10.53s/it]


In [7]:
model.enable_input_require_grads()

In [8]:
messages = [
        {'role':'system','content':'===system_message_test==='},
        {'role':'user','content':'===user_message_test==='},
        {'role':'assistant','content':'===assistant_message_test==='}
    ]

In [9]:
text = tokenizer.apply_chat_template(
        messages,
        tokenize=False, # 表示先不进行分词，然后只返回文本
        add_generation_prompt=True, # 表示在生成文本时，添加一个特殊的标记，用于表示生成的文本开始
        enable_thinking=False # 是否启用思考模式
    )
text

'<|im_start|>system\n===system_message_test===<|im_end|>\n<|im_start|>user\n===user_message_test===<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n===assistant_message_test===<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'

In [10]:
def process_func(example):
    '''
    对数据进行预处理
    :param example: 数据集的一条数据
    :return: 预处理后的数据集
    '''

    # 1.设置数据的最大序列长度，上下文窗口大小
    MAX_LENGTH = 1024

    # 2.初始化返回值列表
    input_ids, attention_mask, labels = [], [], []

    # 3.构建instruction，适配刚才构建好的文本模板格式
    instruction = tokenizer(
        f"<s><|im_start|>system\n现在你要扮演皇帝身边的女人--甄嬛<|im_end|>\n"
        f"<|im_start|>user\n{example['instruction'] + example['input']}<|im_end|>\n"
        f"<|im_start|>assistant\n<think>\n\n</think>\n\n",
        add_special_tokens=False,
    )

    # 4.构建response部分
    response = tokenizer(
        f"{example['output']}",
        add_special_tokens=False,
    )

    # 5.将instruction部分和response部分的input_ids拼接，然后末尾添加pad_token作为结束符
    input_ids = instruction['input_ids'] + response['input_ids'] + [tokenizer.pad_token_id]

    # 6.构建attention_mask，1表示参与计算，0表示不参与计算
    attention_mask = instruction['attention_mask'] + response['attention_mask'] + [1]

    # 7.构建标记labels，-100表示不计算损失
    labels = [-100] * len(instruction['input_ids']) + response['input_ids'] + [tokenizer.pad_token_id]

    # 8.如果序列长度超过上下文窗口大小，则截断
    if len(input_ids) > MAX_LENGTH:
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]

    # 9.返回处理后的数据集
    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'labels': labels
    }

dataset_ds = datas.map(process_func, remove_columns=datas['train'].column_names)

Map: 100%|██████████| 746/746 [00:01<00:00, 552.82 examples/s]


In [11]:
dataset_ds

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2983
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 746
    })
})

In [12]:
datas['train'][10]

{'instruction': '你是谁？', 'input': '', 'output': '我是甄嬛，家父是大理寺少卿甄远道。'}

In [13]:
res = tokenizer.decode(dataset_ds['train'][10]['input_ids'], skip_special_tokens=True)
res

'<s>system\n现在你要扮演皇帝身边的女人--甄嬛\nuser\n你是谁？\nassistant\n<think>\n\n</think>\n\n我是甄嬛，家父是大理寺少卿甄远道。'

In [14]:
from peft import LoraConfig, TaskType, get_peft_model

config = LoraConfig(
        task_type=TaskType.CAUSAL_LM, # 任务类型，因为当前Qwen3-8B本质是一个LLMs，还是基于Decoder-Only架构设计的，所以这里表示任务类型是一个因果语言模型的训练
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'], # 目标模块，这里表示Qwen3-8B模型中的注意力层和MLP层
        inference_mode=False, # 推理模式，False表示训练模式，True表示推理模式
        r=8, # lora低参微调的秩，这里表示低参微调的秩为8
        lora_alpha=32, # lora低参微调的缩放系数，这里表示缩放系数为32
        lora_dropout=0.1, # lora低参微调的dropout概率，这里表示dropout概率为0.1
    )

model = get_peft_model(model, config)

model.print_trainable_parameters()

/usr/local/lib/python3.11/site-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free to reach out:
- X: https://x.com/casper_hansen_
- LinkedIn: https://www.linkedin.com/in/casper-hansen-804005170/

  warnings.warn(_FINAL_DEV_MESSAGE, category=DeprecationWarning, stacklevel=1)


trainable params: 32,112,640 || all params: 14,800,419,840 || trainable%: 0.2170


In [15]:
args = TrainingArguments(
        output_dir=f'./outputs/{ft_model_name}/', # 输出目录
        per_device_train_batch_size=2, # 每个设备的训练批量大小
        per_device_eval_batch_size=2, # 每个设备的验证批量大小
        gradient_accumulation_steps=8, # 梯度累加步数
        logging_steps=10, # 日志打印步数
        num_train_epochs=2, # 训练轮数
        save_steps=10, # 模型保存步数
        learning_rate=1e-4, # 学习率
        save_on_each_node=True, # 每个节点都保存一次模型
        gradient_checkpointing=True # 梯度检查点
    )

In [16]:
trainer = Trainer(
        model=model,
        args=args,
        train_dataset=dataset_ds['train'],
        eval_dataset=dataset_ds['test'],
        data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True)
    )

Detected kernel version 4.15.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


[2025-11-04 12:49:28,949] [INFO] [real_accelerator.py:254:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/usr/local/lib/python3.11/site-packages/deepspeed/ops/op_builder/builder.py:18: DeprecationWarning: The distutils.sysconfig module is deprecated, use sysconfig instead
  import distutils.sysconfig
df: /root/.triton/autotune: 没有那个文件或目录


[2025-11-04 12:49:31,250] [INFO] [logging.py:107:log_dist] [Rank -1] [TorchCheckpointEngine] Initialized with serialization = False


In [ ]:
trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
10,4.358100
20,3.158600
30,2.975800
40,2.983100
